In [1]:
import numpy as np
import pandas as pd
import glob, os
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import scipy.stats as st
import statsmodels.stats.api as sm

from Bio.PDB import PDBParser, PPBuilder, DSSP
from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB.DSSP import make_dssp_dict
from Bio.PDB.Polypeptide import three_to_one

# Make distance maps

## Notes for AlphaFold structures:

Code to save only the high confidence coordinates

```
select high_confidence, b > 70
save ethA_alphaFold_highConf.pdb, high_confidence
```

<!-- 
<ul>
    <li></li>
</ul> -->

In [2]:
# need to leave the Unnamed: 0 index column (don't save with index = False) because evcouplings.compare.distances.py reads in the dataframe with index_col = 0
pncA_structure_coords = np.load("distance_maps/I6XD65.npy")
pncA_distance_map = pd.read_csv("distance_maps/I6XD65.csv")

katG_structure_coords = np.load("distance_maps/P9WIE5.npy")
katG_distance_map = pd.read_csv("distance_maps/P9WIE5.csv")

# check that this is a pairwise matrix, meaning that it's symmetric
assert scipy.linalg.issymmetric(pncA_structure_coords)
assert scipy.linalg.issymmetric(katG_structure_coords)

# and also that the diagonals are all 0
assert sum(np.diagonal(pd.DataFrame(pncA_structure_coords))) == 0
assert sum(np.diagonal(pd.DataFrame(katG_structure_coords))) == 0

In [3]:
# Function to parse CIF file and extract necessary information
def extract_cif_info(cif_file):
    parser = MMCIFParser()
    structure = parser.get_structure('protein', cif_file)
    
    model = structure[0]
    
    # List to hold extracted information
    data = []
    
    # Extract information for each residue
    for chain in model:
        chain_id = chain.id
        for i, res in enumerate(chain):
            if res.id[0] == ' ':  # Exclude heteroatoms for now
                res_id = res.id[1]
                seqres_id = i + 1
                res_name = res.resname
                try:
                    one_letter_code = three_to_one(res_name)
                except KeyError:
                    one_letter_code = 'X'  # Unknown residue
                
                hetatm = res.id[0] != ' '
                coord = res['CA'].coord if 'CA' in res else None

                # # Get secondary structure assignment from DSSP -- not necessary for spatial clustering, and need to install additional dependencies, so skip for now
                # dssp_key = (chain_id, (' ', res_id, ' '))
                # if dssp_key in dssp_dict:
                #     sec_struct = dssp_dict[dssp_key][1]
                #     sec_struct_3state = 'H' if sec_struct in 'GHI' else 'E' if sec_struct == 'E' else 'C'
                # else:
                #     sec_struct = 'NA'
                #     sec_struct_3state = 'NA'
                sec_struct = 'NA'
                sec_struct_3state = 'NA'

                # chain index = 0 because there is only one chain
                # add 1 to len(data) to make it 1-indexed (in residue coordinate space, not index)
                data.append([
                    len(data) + 1, seqres_id, res_id, one_letter_code, res_name,
                    0, chain_id, sec_struct, sec_struct_3state, hetatm, coord
                ])
    
    # Create DataFrame
    columns = ['id', 'seqres_id', 'coord_id', 'one_letter_code',
               'three_letter_code', 'chain_index', 'chain_id', 'sec_struct',
               'sec_struct_3state', 'hetatm', 'coord']
    df = pd.DataFrame(data, columns=columns)
    return df

In [4]:
# mmcif_file = 'ethA_AlphaFold.cif'

# df_ethA_AF = extract_cif_info(mmcif_file)

# # residues 1 and 484-489 are low-confidence in alpha fold, so exclude
# df_ethA_AF_highConf = df_ethA_AF.query("id >= 2 & id <= 483").reset_index(drop=True)
# df_ethA_AF_highConf.to_csv("distance_maps/P9WNF9_AF.csv")

In [5]:
# def calculate_pairwise_distances(coords):
#     coords = np.array([coord for coord in coords if coord is not None])
#     distances = np.linalg.norm(coords[:, np.newaxis] - coords, axis=-1)
#     return distances

In [6]:
# assert sum(pd.isnull(df_ethA_AF_highConf['coord'])) == 0

# # the coordinates are for the alpha carbon atom in each residue
# ca_coords = list(df_ethA_AF_highConf['coord'])
# ca_distances = calculate_pairwise_distances(ca_coords)

# np.save("distance_maps/P9WNF9_AF.npy", ca_distances)

# # pairwise matrix check
# assert scipy.linalg.issymmetric(ca_distances)
# assert sum(np.diagonal(pd.DataFrame(ca_distances))) == 0

# ca_distances.shape

# Results of Clustering on Averaged Fold Change MIC Predictions from Site-Saturation Mutagenesis

In [7]:
def get_significant_GeO_scores(results_dir, pval_thresh=0.05):

    residue_data = pd.read_csv(f"{results_dir}/values_to_cluster.csv")

    GeO_scores = pd.read_csv(f"{results_dir}/G_scores.csv").merge(residue_data, on='residue')
    
    # these are GeO scores for each residue after shuffling the residues
    # permutation test to see if the GeO score for each residue is significantly different from the the null
    GeO_permutation_results = pd.read_csv(f"{results_dir}/random_GeO_iterations_10000.csv.gz", compression="gzip", index_col=[0])
    
    # GeO_pvalues = pd.read_csv(f"{results_dir}/random_GeO_pvalues_10000.csv.gz", compression="gzip")

    for i, row in GeO_scores.iterrows():
    
        residue = row['residue']
        GeO_score = row['G_score']
    
        # what proportion of the permuted Getis-Ord statistics are at least as extreme as the Getis-Ord statistic for a given residue
        if GeO_score > 0:
            pvalue = np.mean(GeO_permutation_results.loc[residue, :].values >= GeO_score)
        else:
            pvalue = np.mean(GeO_permutation_results.loc[residue, :].values <= GeO_score)
    
        GeO_scores.loc[i, "pval"] = pvalue

    _, bh_pvals, _, _ = sm.multipletests(GeO_scores["pval"], method='fdr_bh', is_sorted=False, returnsorted=False)
    _, bonferroni_pvals, _, _ = sm.multipletests(GeO_scores["pval"], method='bonferroni', is_sorted=False, returnsorted=False)
    
    GeO_scores['BH_pval'] = bh_pvals
    GeO_scores['Bonferroni_pval'] = bonferroni_pvals
    
    hot_spots = GeO_scores.query("BH_pval <= @pval_thresh & G_score > 0")
    cold_spots = GeO_scores.query("BH_pval <= @pval_thresh & G_score < 0")

    print(f"{len(hot_spots)} hot spot residues")
    print(f"{len(cold_spots)} cold spot residues")

    return GeO_scores

In [9]:
catalytic_triad = [8, 96, 138]
iron_coordinating = [49, 51, 57, 71]

results_dir = "pncA_SSM_lineage_amino_acid"

pncA_lineage_amino_acid = get_significant_GeO_scores(results_dir)
pncA_lineage_amino_acid.to_csv("../supplement/pncA_GeO_scores.csv", index=False)

24 hot spot residues
14 cold spot residues


In [10]:
pncA_lineage_amino_acid.query("residue in @catalytic_triad")

,residue,G_score,average,pval,BH_pval,Bonferroni_pval
7,8,39.265931,2.640651,0.0067,0.036456,1.000
95,96,46.235978,0.169797,0.0012,0.015654,0.222
137,138,50.461518,0.466222,0.0002,0.007400,0.037


In [11]:
pncA_lineage_amino_acid.query("residue in @iron_coordinating")

,residue,G_score,average,pval,BH_pval,Bonferroni_pval
48,49,35.667616,1.481105,0.0113,0.052262,1.000
50,51,28.672779,2.662728,0.0291,0.090074,1.000
56,57,54.978441,2.370825,0.0002,0.007400,0.037
70,71,32.508500,1.447273,0.0181,0.071245,1.000


In [8]:
results_dir = "katG_SSM_lineage_amino_acid"
katG_lineage_amino_acid = get_significant_GeO_scores(results_dir)
katG_lineage_amino_acid.to_csv("../supplement/katG_GeO_scores.csv", index=False)

102 hot spot residues
193 cold spot residues


In [ ]:
results_dir = "katG_SSM_lineage_amino_acid"
katG_lineage_amino_acid = get_significant_GeO_scores(results_dir)
katG_lineage_amino_acid.to_csv("../supplement/katG_GeO_scores.csv", index=False)

In [15]:
results_dir = 'ethA_SSM_lineage_amino_acid'
ethA_lineage_amino_acid = get_significant_GeO_scores(results_dir)
ethA_lineage_amino_acid.to_csv("../supplement/ethA_GeO_scores.csv", index=False)

53 hot spot residues
18 cold spot residues


From this link: https://www.uniprot.org/uniprotkb/P9WNF9/entry

<ul>
    <li>FAD binding sites: 15, 36, 44-47, 56, 104</li>
    <li>NADP+ binding sites: 54-56, 183-189, 207-208</li>
    <li>Transition state stabilizer: 292</li>
</ul>

In [16]:
# predicted binding site
FAD_binding_sites = [15, 36, 44, 45, 46, 47, 56, 104]
NADP_binding_site = [54, 55, 56, 183, 184, 185, 186, 187, 188, 189, 207, 208]
transition_state_stabilizer = [292]

ethA_lineage_amino_acid.loc[ethA_lineage_amino_acid['residue'].isin(FAD_binding_sites), 'annotation'] = 'FAD binding'
ethA_lineage_amino_acid.loc[ethA_lineage_amino_acid['residue'].isin(NADP_binding_site), 'annotation'] = 'NADP+ binding'
ethA_lineage_amino_acid.loc[ethA_lineage_amino_acid['residue'].isin(transition_state_stabilizer), 'annotation'] = 'transition state stabilizer'

ethA_lineage_amino_acid.query("(residue in @FAD_binding_sites | residue in @NADP_binding_site | residue in @transition_state_stabilizer) & BH_pval <= 0.05").sort_values("G_score", ascending=False)[['residue', 'G_score', 'annotation', 'BH_pval']]

,residue,G_score,annotation,BH_pval
184,186,129.208139,NADP+ binding,0.000000
183,185,121.433521,NADP+ binding,0.004382
182,184,119.940054,NADP+ binding,0.000000
185,187,108.867040,NADP+ binding,0.004382
53,55,102.808628,NADP+ binding,0.012574
54,56,102.373736,NADP+ binding,0.015594
181,183,99.045156,NADP+ binding,0.006886
186,188,96.025761,NADP+ binding,0.013297
187,189,94.374663,NADP+ binding,0.012050
42,44,85.187488,FAD binding,0.025195


In [22]:
ethA_lineage_amino_acid.query("residue==479")

,residue,G_score,average,pval,BH_pval,Bonferroni_pval,annotation
477,479,-46.467931,1.193752,0.0055,0.040167,1.0,nan


In [12]:
def print_pymol_selection_commands(df, pval_thresh=0.05, hot_color='firebrick', cold_color='skyblue', chain_B=False):

    pval_col = 'BH_pval'
    
    # reset the coloring to gray
    print("select all")
    print("color gray80, all\n")

    # hot_spots = [f'A:{num}' for num in ethA_clustering_lineage_amino_acid.query("BH_pval <= @pval_thresh & G_score > 0").residue.values]
    hot_spots = [str(num) for num in df.query(f"{pval_col} <= @pval_thresh & G_score > 0").residue.values]
    hot_spots = '+'.join(hot_spots)

    if len(hot_spots) > 0:
        print(f"select hot_spots, resi {hot_spots} and chain A")
        print(f"color {hot_color}, hot_spots\n")

        if chain_B:
            print(f"select hot_spots, resi {hot_spots} and chain B")
            print(f"color {hot_color}, hot_spots\n")
        
    # hot_spots = [f'A:{num}' for num in ethA_clustering_lineage_amino_acid.query("BH_pval <= @pval_thresh & G_score > 0").residue.values]
    cold_spots = [str(num) for num in df.query(f"{pval_col} <= @pval_thresh & G_score < 0").residue.values]
    cold_spots = '+'.join(cold_spots)

    if len(cold_spots) > 0:
        print(f"select cold_spots, resi {cold_spots} and chain A")
        print(f"color {cold_color}, cold_spots\n")

        if chain_B:
            print(f"select cold_spots, resi {cold_spots} and chain B")
            print(f"color {cold_color}, cold_spots\n")

In [13]:
print_pymol_selection_commands(pncA_lineage_amino_acid)

select all
color gray80, all

select hot_spots, resi 7+8+50+55+56+57+58+59+67+68+96+102+132+134+135+136+137+138+139+140+141+142+143+181 and chain A
color firebrick, hot_spots

select cold_spots, resi 25+26+28+29+30+32+33+34+36+37+38+39+40+85 and chain A
color skyblue, cold_spots



In [14]:
print("select catalytic_triad, resi 8+96+138 and chain A")
print("color magenta, catalytic_triad\n")

print("select iron_coordinating, resi 49+51+57+71 and chain A")
print("color cyan, iron_coordinating")

select catalytic_triad, resi 8+96+138 and chain A
color magenta, catalytic_triad

select iron_coordinating, resi 49+51+57+71 and chain A
color cyan, iron_coordinating


In [13]:
print_pymol_selection_commands(ethA_lineage_amino_acid)

select all
color gray80, all

select hot_spots, resi 44+45+47+48+49+50+51+52+53+54+55+56+57+62+75+146+147+148+162+163+164+165+166+167+180+181+182+183+184+185+186+187+188+189+190+191+206+210+212+292+293+294+295+296+339+340+341+342+343+344+390+391+438 and chain A
color firebrick, hot_spots

select cold_spots, resi 2+3+4+5+6+7+30+31+32+33+96+97+115+129+130+131+132+479 and chain A
color skyblue, cold_spots



In [17]:
print_pymol_selection_commands(katG_lineage_amino_acid, chain_B=True)

select all
color gray80, all

select hot_spots, resi 84+86+87+88+89+90+91+92+93+94+95+96+97+98+99+100+101+102+103+104+105+106+107+108+109+110+111+112+113+120+121+122+123+124+125+126+127+128+129+132+133+134+135+136+137+138+139+140+141+142+143+144+145+146+147+148+149+161+162+165+166+190+227+228+231+232+233+263+265+273+274+275+276+277+278+284+288+296+297+298+299+300+301+302+307+308+309+310+311+312+313+314+315+316+317+318+326+367+368+418+419+420 and chain A
color firebrick, hot_spots

select hot_spots, resi 84+86+87+88+89+90+91+92+93+94+95+96+97+98+99+100+101+102+103+104+105+106+107+108+109+110+111+112+113+120+121+122+123+124+125+126+127+128+129+132+133+134+135+136+137+138+139+140+141+142+143+144+145+146+147+148+149+161+162+165+166+190+227+228+231+232+233+263+265+273+274+275+276+277+278+284+288+296+297+298+299+300+301+302+307+308+309+310+311+312+313+314+315+316+317+318+326+367+368+418+419+420 and chain B
color firebrick, hot_spots

select cold_spots, resi 445+446+447+448+449+450+451+452+45

In [32]:
katG_clustering_amino_acid.query("residue > 640 & residue < 725")

,residue,G_score,average,pval,BH_pval,Bonferroni_pval
617,641,-49.762245,0.016206,0.0001,0.000770,0.0716
618,642,-53.016117,0.022470,0.0000,0.000000,0.0000
619,643,-54.304159,0.017522,0.0001,0.000770,0.0716
620,644,-55.777549,0.008993,0.0000,0.000000,0.0000
621,645,-56.772647,0.005481,0.0000,0.000000,0.0000
...,...,...,...,...,...,...
696,720,-54.546557,-0.025827,0.0000,0.000000,0.0000
697,721,-52.027264,0.005250,0.0001,0.000770,0.0716
698,722,-44.078390,-0.018913,0.0005,0.003255,0.3580
699,723,-44.079597,-0.001353,0.0010,0.006017,0.7160


In [11]:
print_pymol_selection_commands(ethA_clustering_lineage_amino_acid, pval_thresh=0.05)

select all
color gray80, all

select hot_spots, resi 44+45+46+47+48+49+50+51+52+53+54+55+56+57+62+75+78+146+147+148+162+163+164+165+166+167+180+181+182+183+184+185+186+187+188+189+190+191+206+210+212+292+293+294+295+339+340+341+342+343+344+390+391+438 and chain A
color firebrick, hot_spots

select cold_spots, resi 2+3+4+5+6+7+29+30+31+32+33+96+97+113+115+129+130+131+132+479 and chain A
color skyblue, cold_spots


In [ ]:
print_pymol_selection_commands(katG_clustering_amino_acid, pval_thresh=0.05, chain_B=True)